# HEILO Seed Code v0.1: modelo próprio de código (Colab)

Este notebook treina **do zero** um modelo de código só da HEILO: ~30 M parâmetros, com tokenizer próprio, lendo código Python público (codeparrot-clean). Ele não usa o Qwen nem nenhum outro modelo.

**Expectativa honesta:** ele aprende sintaxe e padrões comuns do Python, e deve conseguir **autocompletar funções simples**. Não entende pedidos complexos e não programa como um especialista.

**Avaliação real:**
- 20 funções com docstring. O modelo escreve o corpo e **os testes são executados de verdade**, medindo pass@1 e pass@6.
- % de código válido.
- Perplexidade em código de validação que ele nunca viu.

Pode rodar **ao mesmo tempo** que o notebook do HEILO Seed v1.0: usa outra pasta do Drive (`heilo_seed_code`) e outra pasta do projeto (`models/seed_code/`).

**Antes:** `Ambiente de execução → Alterar tipo → GPU` e o secret `GITHUB_TOKEN` (o mesmo do outro notebook). Depois, `Executar tudo`. Leva de 2h30 a 3h.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v0.1"
MINUTOS_PRETREINO = 150
MAX_CARACTERES = 500_000_000   # ~500 MB de código Python
VOCAB = 16000
LOTE = 32
import torch, time, os, sys, json, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive (checkpoints; retoma se o Colab cair)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_seed_code"
os.makedirs(PASTA, exist_ok=True)

In [ ]:
# 3) Código da HEILO (clonado numa pasta própria para não esbarrar no outro notebook)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
BASE = "/content/code_run"
if not os.path.exists(f"{BASE}/heilo"):
    os.makedirs(BASE, exist_ok=True)
    subprocess.run(["git", "clone", "-b", BRANCH, URL, f"{BASE}/heilo"], check=True)
sys.path.insert(0, BASE)
!pip -q install datasets tokenizers

In [ ]:
# 4) Tokenizer de código próprio + corpus tokenizado (salvos no Drive)
from pathlib import Path
from heilo.training import seed_code, pretrain
from heilo.models.seed.tokenizer import BPETokenizer
tok_arq, corpus_dir = Path(PASTA) / "tokenizer.json", Path(PASTA) / "corpus"
if tok_arq.exists():
    tok = BPETokenizer.from_json(tok_arq.read_text(encoding="utf-8"))
else:
    t0 = time.time()
    amostra = list(seed_code.textos_python(80_000_000))
    tok = BPETokenizer.treinar(iter(amostra), vocab_size=VOCAB)
    tok_arq.write_text(tok.tok.to_str(), encoding="utf-8")
    del amostra
    print(f"tokenizer treinado em {time.time()-t0:.0f}s")
print("teste:", tok.tok.encode("def soma(a, b):\n    return a + b").tokens)
if not (corpus_dir / "meta.json").exists():
    t0 = time.time()
    meta = pretrain.preparar_corpus(seed_code.textos_python(MAX_CARACTERES), tok, corpus_dir,
                                    max_tokens=10**12, log=lambda s: None)
    print(meta, f"{(time.time()-t0)/60:.1f} min")
else:
    print(open(corpus_dir / "meta.json").read())

In [ ]:
# 5) Velocidade → passos que cabem no tempo
cfg = seed_code.CONFIG_CODE
ck = Path(PASTA) / "pretreino.pt"
if not Path(PASTA, "passos.txt").exists():
    t0 = time.time()
    pretrain.pretreinar(cfg, tok, corpus_dir, Path("/content/vel_code.pt"), passos=60, lote=LOTE,
                        avaliar_cada=10**9, salvar_cada=10**9, aquecimento=10, log=lambda s: None)
    seg = (time.time() - t0) / 60
    Path(PASTA, "passos.txt").write_text(str(int(MINUTOS_PRETREINO * 60 / seg)))
PASSOS = int(Path(PASTA, "passos.txt").read_text())
print(PASSOS, "passos")

In [ ]:
# 6) Pré-treino em código (retoma sozinho)
res = pretrain.pretreinar(cfg, tok, corpus_dir, ck, passos=PASSOS, lote=LOTE,
                          tempo_max_min=MINUTOS_PRETREINO + 20, avaliar_cada=500, salvar_cada=1000)
modelo = res["modelo"]
print("pré-treino:", res["passo"], "passos |", res["hist"][-1] if res["hist"] else "")

In [ ]:
# 7) Avaliação REAL: completar funções e executar os testes
av = seed_code.avaliar_codigo(modelo)
print({k: v for k, v in av.items() if k != "detalhes"})
for d in av["detalhes"][:4]:
    print("-" * 40); print(d["codigo"])

In [ ]:
# 8) Exportar + registrar + salvar no GitHub
from heilo.training.records import agora
destino = Path(BASE) / "heilo" / "models" / "seed_code" / "versions" / NOME_VERSAO
destino.mkdir(parents=True, exist_ok=True)
pretrain.exportar(modelo, destino / "heilo_seed_code.pt",
                  {"versao": NOME_VERSAO, "passos": res["passo"], "hist": res["hist"],
                   "corpus": json.loads((corpus_dir / "meta.json").read_text())})
manifesto = {"versao": NOME_VERSAO, "criada": agora(), "tipo": "HEILO Seed Code (do zero)",
             "config": cfg.__dict__, "passos": res["passo"], "minutos": res["minutos"],
             "ultima_validacao": res["hist"][-1] if res["hist"] else None,
             "avaliacao": {k: v for k, v in av.items() if k != "detalhes"},
             "corpus": "codeparrot/codeparrot-clean (código Python público do GitHub; licenças variadas)"}
(destino / "manifest.json").write_text(json.dumps(manifesto, indent=2, ensure_ascii=False))
(destino / "avaliacao.json").write_text(json.dumps(av, indent=2, ensure_ascii=False))
print(json.dumps(manifesto, indent=2, ensure_ascii=False)[:1500])
!cd {BASE}/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd {BASE}/heilo && git add -f models/seed_code && git commit -m "HEILO Seed Code {NOME_VERSAO} (Colab)" || echo nada
!cd {BASE}/heilo && git pull --rebase && git push